# Exploring Epsilon-Neighborhoods in DBSCAN> **Difficulty**: Intermediate  > **Estimated Time**: 30-40 minutes  > **Prerequisites**: Basic understanding of DBSCAN concepts, completed 02_mathematical_foundations.ipynb## Learning ObjectivesBy the end of this notebook, you will be able to:- Visualize epsilon-neighborhoods (ε-neighborhoods) interactively- Understand how epsilon affects neighborhood size and cluster formation- Explore the relationship between epsilon and point density- Identify optimal epsilon values through visual exploration- Understand the impact of epsilon on clustering results## Paper ReferencesThis notebook explores concepts from:- **Section 4.1** (p. 227): Density-Based Notions of Clusters - ε-neighborhood definition- **Section 5.1** (p. 229): Determining the Parameters - epsilon selection strategies**Full Reference:**  Ester, M., Kriegel, H. P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. In *KDD* (Vol. 96, No. 34, pp. 226-231).## Table of Contents1. [Setup and Imports](#setup)2. [Understanding ε-Neighborhoods](#understanding)3. [Interactive Epsilon Exploration](#interactive)4. [Effect on Clustering Results](#clustering-effect)5. [Density Variations](#density)6. [Practical Guidelines](#guidelines)7. [Summary](#summary)

---## 1. Setup and Imports <a id='setup'></a>First, let's import the necessary libraries and set up our environment.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport ipywidgets as widgetsfrom IPython.display import displayimport syssys.path.append('..')from src.dbscan_from_scratch import DBSCANfrom src.visualization import DBSCANVisualizerfrom sklearn.datasets import make_moons, make_blobs# Set random seed for reproducibilitynp.random.seed(42)# Initialize visualizerviz = DBSCANVisualizer()print("✓ Setup complete!")

---## 2. Understanding ε-Neighborhoods <a id='understanding'></a>### Mathematical Definition [Paper §4.1, p. 227]The **ε-neighborhood** of a point p is defined as:```N_ε(p) = {q ∈ D | dist(p, q) ≤ ε}```Where:- **D** is the dataset- **dist(p, q)** is the distance between points p and q- **ε** (epsilon) is the radius parameter**Intuition**: The ε-neighborhood is simply all points within distance ε from point p. In 2D space, this forms a circle of radius ε centered at p.### Why ε-Neighborhoods MatterThe ε-neighborhood is the foundation of DBSCAN:- **Core points** are identified by having enough neighbors in their ε-neighborhood- **Clusters** are formed by connecting points through overlapping ε-neighborhoods- **Noise points** have sparse ε-neighborhoodsLet's visualize this concept:

In [ ]:
# Generate sample dataX, _ = make_moons(n_samples=100, noise=0.05, random_state=42)# Select a point to explorepoint_idx = 20eps = 0.3# Visualize the epsilon-neighborhoodviz.plot_epsilon_neighborhood(X, point_idx, eps)plt.show()# Calculate neighborhood sizedistances = np.sqrt(np.sum((X - X[point_idx])**2, axis=1))neighbors = np.where(distances <= eps)[0]print(f"\n📊 Neighborhood Statistics:")print(f"  • Selected point: {point_idx}")print(f"  • Epsilon (ε): {eps}")print(f"  • |N_ε(p)|: {len(neighbors)} points")print(f"  • Includes the point itself: {point_idx in neighbors}")

---## 3. Interactive Epsilon Exploration <a id='interactive'></a>Now let's explore how changing epsilon affects the neighborhood size interactively. Use the sliders below to:- Change the **epsilon** value to see how the neighborhood grows or shrinks- Select different **points** to explore their neighborhoods### Interactive Widget

In [ ]:
# Create interactive exploration widgetdef explore_epsilon_neighborhood(epsilon, point_index):    # Ensure point_index is within bounds    point_index = min(max(0, point_index), len(X) - 1)        # Calculate neighborhood    distances = np.sqrt(np.sum((X - X[point_index])**2, axis=1))    neighbors = np.where(distances <= epsilon)[0]        # Visualize    viz.plot_epsilon_neighborhood(X, point_index, epsilon)    plt.show()        # Print statistics    print(f"\n📊 Neighborhood Statistics:")    print(f"  • Point index: {point_index}")    print(f"  • Point coordinates: ({X[point_index, 0]:.3f}, {X[point_index, 1]:.3f})")    print(f"  • Epsilon (ε): {epsilon:.2f}")    print(f"  • |N_ε(p)|: {len(neighbors)} points")    print(f"  • Neighborhood density: {len(neighbors) / (np.pi * epsilon**2):.2f} points/unit²")        # Determine if this would be a core point with min_pts=5    min_pts = 5    is_core = len(neighbors) >= min_pts    print(f"\n🎯 Point Classification (with MinPts={min_pts}):")    print(f"  • Would be core point: {is_core}")    if is_core:        print(f"  • Has {len(neighbors)} >= {min_pts} neighbors ✓")    else:        print(f"  • Has {len(neighbors)} < {min_pts} neighbors ✗")# Create interactive widgetepsilon_slider = widgets.FloatSlider(    value=0.3, min=0.1, max=1.0, step=0.05,    description='Epsilon (ε):', continuous_update=False, readout_format='.2f')point_slider = widgets.IntSlider(    value=20, min=0, max=len(X) - 1, step=1,    description='Point Index:', continuous_update=False)# Display interactive widgetwidgets.interactive(explore_epsilon_neighborhood, epsilon=epsilon_slider, point_index=point_slider)

---## 4. Effect on Clustering Results <a id='clustering-effect'></a>Now let's see how epsilon affects the final clustering results. We'll run DBSCAN with different epsilon values and compare the outcomes.### Comparing Different Epsilon Values

In [ ]:
# Test different epsilon valueseps_values = [0.15, 0.3, 0.5, 0.8]min_pts = 5fig, axes = plt.subplots(2, 2, figsize=(14, 12))axes = axes.flatten()for idx, eps in enumerate(eps_values):    # Run DBSCAN    dbscan = DBSCAN(eps=eps, min_pts=min_pts)    labels = dbscan.fit_predict(X)        # Count clusters and noise    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)    n_noise = list(labels).count(-1)    n_core = len(dbscan.get_core_points())        # Plot    ax = axes[idx]    unique_labels = set(labels)    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))        for label, color in zip(unique_labels, colors):        if label == -1:            mask = labels == label            ax.scatter(X[mask, 0], X[mask, 1], c='black', marker='x',                       s=50, alpha=0.5, label='Noise')        else:            mask = labels == label            ax.scatter(X[mask, 0], X[mask, 1], c=[color], s=50,                       alpha=0.7, label=f'Cluster {label}')        ax.set_title(f'ε = {eps:.2f}\n{n_clusters} clusters, {n_noise} noise, {n_core} core points',                 fontsize=11, fontweight='bold')    ax.set_xlabel('Feature 1')    ax.set_ylabel('Feature 2')    ax.grid(True, alpha=0.3)    ax.legend(loc='best', fontsize=8)plt.tight_layout()plt.show()print("\n🔍 Analysis of Epsilon Effects:")print("\nε = 0.15 (Too Small):")print("  • Many small clusters or all noise")print("  • Neighborhoods too small to capture cluster structure")print("\nε = 0.3 (Just Right):")print("  • Correctly identifies the two moon-shaped clusters")print("  • Minimal noise points")print("\nε = 0.5 (Getting Large):")print("  • May start merging distinct clusters")print("\nε = 0.8 (Too Large):")print("  • All points likely in one cluster")

---## 7. Summary <a id='summary'></a>### Key Takeaways1. **ε-Neighborhood Definition** [Paper §4.1, p. 227]:   - N_ε(p) = {q ∈ D | dist(p, q) ≤ ε}   - All points within distance ε from point p   - Forms a circle (2D) or hypersphere (higher dimensions)2. **Epsilon's Role in DBSCAN**:   - Defines what "nearby" means   - Determines neighborhood size for core point classification   - Critical parameter affecting cluster formation   - Works together with MinPts to define density3. **Effects of Epsilon Size**:   - **Too small**: Many small clusters, lots of noise   - **Too large**: Clusters merge, everything becomes one cluster   - **Just right**: Captures true cluster structure4. **Selection Strategies** [Paper §5.1, p. 229]:   - **K-distance graph method**: Most reliable automated approach   - Look for the "elbow" in sorted k-distances   - Consider domain knowledge and data characteristics### What You Can Do Now✓ Visualize and understand ε-neighborhoods  ✓ Explore epsilon effects interactively  ✓ Use k-distance graph for epsilon selection  ✓ Recognize when epsilon is too small or too large  ✓ Understand epsilon's relationship with density  ---## Next Steps1. **[04_density_concepts.ipynb](04_density_concepts.ipynb)** - Explore density-reachability and density-connectivity2. **[05_algorithm_walkthrough.ipynb](05_algorithm_walkthrough.ipynb)** - See the algorithm in action step-by-step3. **[06_parameter_tuning.ipynb](06_parameter_tuning.ipynb)** - Advanced parameter selection techniques---**Happy Exploring! 🔍**